# 01 - Data Analysis (EDA)
Goal: understand the data BEFORE modelling. Every number here comes from your own file.

In [ ]:
import sys; sys.path.insert(0, '..')
import warnings; warnings.filterwarnings('ignore')
import pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from collections import Counter
from src import config
from src.data import load_raw, prepare

## Load, first 5 rows, shape

In [ ]:
raw = load_raw()
print('Shape:', raw.shape)
raw.head()

## Missing values and duplicates

In [ ]:
print(raw.isna().sum())
print('Duplicate review texts:', raw.duplicated(subset=[config.RAW_TEXT_COL]).sum())

## Class distribution
`CG` = computer-generated (fake), `OR` = original (genuine).

In [ ]:
print(raw[config.RAW_LABEL_COL].value_counts())
print(raw[config.RAW_LABEL_COL].value_counts(normalize=True).round(3))
sns.countplot(data=raw, x=config.RAW_LABEL_COL)
plt.title('Class distribution'); plt.show()

If the two bars are nearly equal the data is **balanced**. If one class is much bigger, accuracy can look good while the minority class is ignored (class imbalance).

## Review length

In [ ]:
df = prepare(raw)
df['n_words'] = df[config.TEXT_COL].str.split().str.len()
sns.histplot(data=df, x='n_words', hue=config.TARGET_COL, bins=50, element='step')
plt.title('Review length (words), 0=genuine 1=fake'); plt.show()
df.groupby(config.TARGET_COL)['n_words'].describe()

## Most frequent words per class

In [ ]:
from src.preprocess import clean_text
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS as SW
def top_words(s, n=15):
    c = Counter(w for t in s for w in clean_text(t).split() if w not in SW and len(w) > 2)
    return c.most_common(n)
for label, name in [(0, 'genuine'), (1, 'fake')]:
    print(name, top_words(df.loc[df[config.TARGET_COL] == label, config.TEXT_COL]))